<a href="https://colab.research.google.com/github/AmandineBaranger/Group_06_DS_Project/blob/feat%2Fbaseline-notebook/notebooks/02_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diabetes 130-US Hospitals - Baseline Models and Evaluation

- Part 1 (sections 1 to 3): we load the prepared train and test sets and we define our validation strategy grouped by patient.
- Part 2 (sections 4 to 7): we build and evaluate three baseline models (Dummy Classifier, Logistic Regression, and Decision Tree) using ROC-AUC, PR-AUC, Recall, Precision, and F1-score.
- Part 3 (sections 8 to 9): we compare all baselines on the validation folds, check error patterns, and save our reference results for Week 3.

## 1. Imports and paths

In [3]:
# import os and sys to handle paths and import our own module from src
import os
import sys
import urllib.request
# import numpy and pandas for data manipulation
import numpy as np
import pandas as pd
# import matplotlib and seaborn for charts
import matplotlib.pyplot as plt
import seaborn as sns

# import scikit-learn tools for grouped cross-validation and baseline models
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

# use a clean style for all charts
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

# paths to raw data on GitHub and local src folder
RAW_PATH = "https://raw.githubusercontent.com/AmandineBaranger/Group_06_DS_Project/main/data/raw/diabetic_data.csv"
SRC_DIR = os.path.join("..", "src")
SEED = 42

# if running in Google Colab without local repo files, fetch src/preprocessing.py from GitHub
os.makedirs(SRC_DIR, exist_ok=True)
prep_file = os.path.join(SRC_DIR, "preprocessing.py")
if not os.path.exists(prep_file):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/AmandineBaranger/Group_06_DS_Project/main/src/preprocessing.py",
        prep_file
    )

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

import preprocessing

What this code does: we import the required libraries, set a fixed random seed (`SEED = 42`), and define relative paths to load the preprocessed train and test datasets created in the first module. We also define the GitHub raw URL as a fallback so the notebook runs seamlessly both on local Jupyter setups and on Google Colab.

## 2. Loading the processed train and test sets

In [5]:
# load raw dataset from GitHub
df = pd.read_csv(RAW_PATH, keep_default_na=False, na_values=["?"], low_memory=False)

# apply the exact same cleaning steps as Part 1 of notebook 01
df = df.drop(columns=["weight", "payer_code"])
df["medical_specialty"] = df["medical_specialty"].fillna("Missing")
df["race"] = df["race"].fillna("Unknown")
df = df.drop_duplicates()
df = df[df["gender"] != "Unknown/Invalid"]
df = df[~df["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]

nunique = df.nunique()
top_share = df.apply(lambda s: s.value_counts(normalize=True).iloc[0])
to_drop = sorted(set(nunique[nunique == 1].index.tolist() + top_share[(top_share >= 0.995) & (nunique > 1)].index.tolist()))
df = df.drop(columns=to_drop)

# create binary target (<30 days readmission) and drop identifiers/target
df["target"] = (df["readmitted"] == "<30").astype(int)
groups = df["patient_nbr"]
y = df["target"]
X = df.drop(columns=["readmitted", "encounter_id", "target", "patient_nbr"])

# apply feature engineering from src/preprocessing.py
X = preprocessing.prepare_features(X)

# split by patient (80% train / 20% test) using StratifiedGroupKFold
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train_raw, X_test_raw = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)
groups_train = groups.iloc[train_idx].reset_index(drop=True)

# fit the preprocessor strictly on X_train to avoid any data leakage
num_cols, cat_cols = preprocessing.get_column_types(X_train_raw)
preprocessor = preprocessing.build_preprocessor(num_cols, cat_cols)

X_train_arr = preprocessor.fit_transform(X_train_raw)
X_test_arr = preprocessor.transform(X_test_raw)

# convert back to DataFrames with clean feature names
try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    feature_names = [f"feature_{i}" for i in range(X_train_arr.shape[1])]

X_train = pd.DataFrame(X_train_arr, columns=feature_names)
X_test = pd.DataFrame(X_test_arr, columns=feature_names)

# verify shapes and target distribution
print("X_train shape :", X_train.shape)
print("X_test shape  :", X_test.shape)
print("Train positive rate (<30 days) :", round(y_train.mean(), 4))
print("Test positive rate (<30 days)  :", round(y_test.mean(), 4))
print("Unique patients in train       :", groups_train.nunique())

X_train shape : (79309, 138)
X_test shape  : (20031, 138)
Train positive rate (<30 days) : 0.1134
Test positive rate (<30 days)  : 0.1158
Unique patients in train       : 56036


In [2]:
# helper function to load from local processed folder first, or from GitHub if on Colab
def load_processed_csv(filename):
    local_path = os.path.join(PROCESSED_DIR, filename)
    if os.path.exists(local_path):
        return pd.read_csv(local_path)
    return pd.read_csv(f"{GITHUB_PROCESSED_URL}/{filename}")

# load training and test features and targets produced by Part 2 of notebook 01
X_train = load_processed_csv("X_train.csv")
X_test = load_processed_csv("X_test.csv")
y_train = load_processed_csv("y_train.csv").squeeze()
y_test = load_processed_csv("y_test.csv").squeeze()

# load patient identifiers for the training set to prevent patient leakage during cross-validation
groups_train = load_processed_csv("groups_train.csv").squeeze()

# print dataset shapes and target proportions to verify consistency
print("X_train shape :", X_train.shape)
print("X_test shape  :", X_test.shape)
print("Train positive rate (<30 days) :", round(y_train.mean(), 4))
print("Test positive rate (<30 days)  :", round(y_test.mean(), 4))
print("Unique patients in train       :", groups_train.nunique())

HTTPError: HTTP Error 404: Not Found